# 🗺️ Rumsey Map OCR — Detection Model Fine-tuning
Run each cell **top to bottom**. Training takes **4-6 hours** on a T4 GPU.

> **Before running:** `Runtime → Change runtime type → T4 GPU`

### What you need on Google Drive first:
```
My Drive/
└── Rumsey_OCR/
    ├── train_data/
    │   └── det/            ← upload this from your laptop
    │       ├── train_list.txt
    │       ├── val_list.txt
    │       └── train_images/
    └── det_icdar_finetune.yml   ← upload this config file
```


## Step 1 — Verify GPU


In [1]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout or '❌ No GPU — go to Runtime → Change runtime type → T4 GPU')


Mon Feb 23 15:01:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Mount Google Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Rumsey_OCR'

# Verify required files are on Drive
checks = {
    'Training data':  os.path.join(DRIVE_ROOT, 'train_data/det/train_list.txt'),
    'Val data':       os.path.join(DRIVE_ROOT, 'train_data/det/val_list.txt'),
    'Config file':    os.path.join(DRIVE_ROOT, 'det_icdar_finetune.yml'),
}
all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌  MISSING'
    print(f'  {status}  {name}: {path}')
    if not exists: all_ok = False
if not all_ok:
    print('\n⚠️  Upload missing files to Drive before continuing!')
else:
    print('\n✅ All required files found!')


Mounted at /content/drive
  ✅  Training data: /content/drive/MyDrive/Rumsey_OCR/train_data/det/train_list.txt
  ✅  Val data: /content/drive/MyDrive/Rumsey_OCR/train_data/det/val_list.txt
  ✅  Config file: /content/drive/MyDrive/Rumsey_OCR/det_icdar_finetune.yml

✅ All required files found!


## Step 3 — Install PaddlePaddle GPU
*(Takes ~2 min — run once per session)*


In [4]:
# Install PaddlePaddle - auto-detects correct CUDA version
import subprocess

# Check CUDA version
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

# Install the matching PaddlePaddle GPU build
!pip install paddlepaddle-gpu==2.6.1 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q

import paddle
print(f'PaddlePaddle: {paddle.__version__}')
print(f'GPU available: {paddle.is_compiled_with_cuda()}')

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.8/758.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.5 MB/s eta 0:00:00
PaddlePaddle: 2.6.1
GPU available: True


## Step 4 — Get PaddleOCR Toolkit
Clones the official PaddleOCR training tools. *(No need to upload — pulled from PaddlePaddle directly)*


In [5]:
import os

PADDLEOCR_DIR = '/content/PaddleOCR'

if not os.path.exists(PADDLEOCR_DIR):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git {PADDLEOCR_DIR} -q
    print('Cloned PaddleOCR')
else:
    print('PaddleOCR already present')

# Install requirements
!pip install -r {PADDLEOCR_DIR}/requirements.txt -q
print('Requirements installed!')


Cloned PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 107.8 MB/s eta 0:00:00
Requirements installed!


## Step 5 — Setup Workspace


In [6]:
import os, shutil

DRIVE_ROOT   = '/content/drive/MyDrive/Rumsey_OCR'
PADDLEOCR_DIR = '/content/PaddleOCR'
WORK_DIR      = '/content/workspace'
os.makedirs(WORK_DIR, exist_ok=True)

# Link training data from Drive into workspace
DET_DATA_SRC = os.path.join(DRIVE_ROOT, 'train_data/det')
DET_DATA_DST = os.path.join(WORK_DIR, 'train_data/det')
os.makedirs(os.path.dirname(DET_DATA_DST), exist_ok=True)
if not os.path.exists(DET_DATA_DST):
    os.symlink(DET_DATA_SRC, DET_DATA_DST)
    print(f'Linked train_data/det from Drive')

# Copy config into workspace (and fix paths to be absolute)
CONFIG_SRC = os.path.join(DRIVE_ROOT, 'det_icdar_finetune.yml')
CONFIG_DST = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
shutil.copy2(CONFIG_SRC, CONFIG_DST)

# Link output directory to Drive for REAL-TIME checkpoint saving
# This ensures that if Colab disconnects, your progress is saved on Drive!
DRIVE_OUT_DIR = os.path.join(DRIVE_ROOT, 'output/det_finetune')
LOCAL_OUT_DIR = os.path.join(WORK_DIR, 'output/det_finetune')
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_OUT_DIR), exist_ok=True)

if not os.path.exists(LOCAL_OUT_DIR):
    os.symlink(DRIVE_OUT_DIR, LOCAL_OUT_DIR)
    print(f'✅ Output directory symlinked to Drive: {DRIVE_OUT_DIR}')

# Patch paths in config to be absolute
with open(CONFIG_DST) as f:
    cfg = f.read()
cfg = cfg.replace('../train_data/det', DET_DATA_DST)
cfg = cfg.replace('../output/det_finetune', LOCAL_OUT_DIR)
with open(CONFIG_DST, 'w') as f:
    f.write(cfg)
print('Config ready at:', CONFIG_DST)
print('Workspace ready!')


Linked train_data/det from Drive
✅ Output directory symlinked to Drive: /content/drive/MyDrive/Rumsey_OCR/output/det_finetune
Config ready at: /content/workspace/det_icdar_finetune.yml
Workspace ready!


## Step 6 — Download Pretrained Detection Model


In [7]:
import os, subprocess
WORK_DIR    = '/content/workspace'
PRETRAINED  = os.path.join(WORK_DIR, 'pretrained/ch_PP-OCRv4_det_train')
TAR_FILE    = os.path.join(WORK_DIR, 'pretrained/ch_PP-OCRv4_det_train.tar')
os.makedirs(os.path.join(WORK_DIR, 'pretrained'), exist_ok=True)

if not os.path.exists(PRETRAINED):
    # Remove any partial download
    if os.path.exists(TAR_FILE): os.remove(TAR_FILE)

    URL = 'https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_train.tar'
    print(f'Downloading from: {URL}')

    # Use curl with progress bar (more reliable than wget in Colab)
    result = subprocess.run(
        ['curl', '-L', '--progress-bar', '-o', TAR_FILE, URL],
        check=False
    )

    size = os.path.getsize(TAR_FILE) if os.path.exists(TAR_FILE) else 0
    print(f'Downloaded file size: {size / 1024 / 1024:.1f} MB')

    if size > 10_000_000:  # >10MB means it's real
        !tar -xf {TAR_FILE} -C {os.path.join(WORK_DIR, 'pretrained')}
        print('✅ Extracted!')
    else:
        print('❌ File too small — possible wrong URL or network issue')
        # Show what was downloaded
        !head -c 200 {TAR_FILE}
else:
    print('✅ Already present')

# Update config
CONFIG_DST = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
if os.path.exists(CONFIG_DST):
    with open(CONFIG_DST) as f: cfg = f.read()
    cfg = cfg.replace('../pretrained/ch_PP-OCRv4_det_train/best_accuracy',
                      os.path.join(PRETRAINED, 'best_accuracy'))
    with open(CONFIG_DST, 'w') as f: f.write(cfg)
    print('Config updated')

Downloaded file size: 13.6 MB
✅ Extracted!
Config updated


## Step 7 — Verify Training Data


In [8]:
WORK_DIR = '/content/workspace'
import os

for fname in ['train_list.txt', 'val_list.txt']:
    path = os.path.join(WORK_DIR, 'train_data/det', fname)
    if os.path.exists(path):
        with open(path) as f: n = sum(1 for _ in f)
        print(f'✅ {fname}: {n:,} images')
    else:
        print(f'❌ MISSING: {path}')


✅ train_list.txt: 200 images
✅ val_list.txt: 40 images


In [ ]:
# ============================================================
# MASTER PATCH CELL — Run after every Colab reconnect,
# BEFORE running Step 8 (Training)
# ============================================================

# --- PATCH 1: det_basic_loss.py ---
content_basic = '''from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import math
import paddle
from paddle import nn
import paddle.nn.functional as F


def _safe(t):
    t = paddle.where(paddle.isnan(t), paddle.zeros_like(t), t)
    t = paddle.where(paddle.isinf(t), paddle.zeros_like(t), t)
    return t

def _clean_gt(t):
    """Clean ground truth: remove NaN/Inf and clamp to [0,1]."""
    t = paddle.where(paddle.isnan(t), paddle.zeros_like(t), t)
    t = paddle.where(paddle.isinf(t), paddle.zeros_like(t), t)
    return paddle.clip(t, min=0.0, max=1.0)


class BalanceLoss(nn.Layer):
    def __init__(self, balance_loss=True, main_loss_type="DiceLoss",
                 negative_ratio=3, return_origin=False, eps=1e-6, **kwargs):
        super(BalanceLoss, self).__init__()
        self.balance_loss = balance_loss
        self.main_loss_type = main_loss_type
        self.negative_ratio = negative_ratio
        self.return_origin = return_origin
        self.eps = eps
        if self.main_loss_type == "CrossEntropy":
            self.loss = nn.CrossEntropyLoss()
        elif self.main_loss_type == "Euclidean":
            self.loss = nn.MSELoss()
        elif self.main_loss_type == "DiceLoss":
            self.loss = DiceLoss(self.eps)
        elif self.main_loss_type == "BCELoss":
            self.loss = BCELoss(reduction="none")
        elif self.main_loss_type == "MaskL1Loss":
            self.loss = MaskL1Loss(self.eps)
        else:
            raise Exception("main_loss_type must be one of [CrossEntropy, DiceLoss, Euclidean, BCELoss, MaskL1Loss]")

    def forward(self, pred, gt, mask=None):
        pred = _safe(pred)
        gt   = _clean_gt(gt)
        if mask is not None:
            mask = _clean_gt(mask)

        positive = gt * mask
        negative = (1 - gt) * mask

        try:
            pos_val = float(positive.sum().item())
            positive_count = 0 if (math.isnan(pos_val) or math.isinf(pos_val) or pos_val > 1e9) else max(0, int(pos_val))
        except (ValueError, OverflowError):
            positive_count = 0

        try:
            neg_val = float(negative.sum().item())
            neg_val = 0.0 if (math.isnan(neg_val) or math.isinf(neg_val) or neg_val > 1e9) else neg_val
            negative_count = max(0, int(min(neg_val, positive_count * self.negative_ratio)))
        except (ValueError, OverflowError):
            negative_count = 0

        loss = self.loss(pred, gt, mask=mask)
        loss = _safe(loss)

        if not self.balance_loss:
            return loss

        positive_loss = _safe(positive * loss)
        negative_loss = _safe(negative * loss)
        negative_loss = paddle.reshape(negative_loss, shape=[-1])

        negative_count = min(negative_count, int(negative_loss.shape[0]))

        if positive_count == 0:
            balance_loss = paddle.to_tensor(0.0)
        elif negative_count > 0:
            sort_loss = negative_loss.sort(descending=True)
            negative_loss = sort_loss[:negative_count]
            balance_loss = (positive_loss.sum() + negative_loss.sum()) / (
                positive_count + negative_count + self.eps)
        else:
            balance_loss = positive_loss.sum() / (positive_count + self.eps)

        balance_loss = _safe(balance_loss)
        if self.return_origin:
            return balance_loss, loss
        return balance_loss


class DiceLoss(nn.Layer):
    def __init__(self, eps=1e-6):
        super(DiceLoss, self).__init__()
        self.eps = eps

    def forward(self, pred, gt, mask, weights=None):
        pred = _safe(F.sigmoid(pred))
        gt   = _clean_gt(gt)
        mask = _clean_gt(mask)
        if weights is not None:
            mask = weights * mask
        intersection = paddle.sum(pred * gt * mask)
        union = paddle.sum(pred * mask) + paddle.sum(gt * mask) + self.eps
        loss = 1 - 2.0 * intersection / union
        return paddle.clip(_safe(loss), max=1.0)


class MaskL1Loss(nn.Layer):
    def __init__(self, eps=1e-6):
        super(MaskL1Loss, self).__init__()
        self.eps = eps

    def forward(self, pred, gt, mask):
        gt   = _clean_gt(gt)
        mask = _clean_gt(mask)
        loss = (paddle.abs(pred - gt) * mask).sum() / (mask.sum() + self.eps)
        return _safe(paddle.mean(loss))


class BCELoss(nn.Layer):
    def __init__(self, reduction="mean"):
        super(BCELoss, self).__init__()
        self.reduction = reduction

    def forward(self, input, label, mask=None, weight=None, name=None):
        input = paddle.clip(input, min=1e-6, max=1 - 1e-6)
        label = _clean_gt(label)
        loss = F.binary_cross_entropy(input, label, reduction=self.reduction)
        return _safe(loss)
'''

with open('/content/PaddleOCR/ppocr/losses/det_basic_loss.py', 'w') as f:
    f.write(content_basic)
print('✅ Patch 1/5: det_basic_loss.py')

# --- PATCH 2: det_db_loss.py ---
content_db = '''from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import paddle
from paddle import nn

from .det_basic_loss import BalanceLoss, MaskL1Loss, DiceLoss


class DBLoss(nn.Layer):
    def __init__(self, balance_loss=True, main_loss_type="DiceLoss",
                 alpha=5, beta=10, ohem_ratio=3, eps=1e-6, **kwargs):
        super(DBLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.dice_loss = DiceLoss(eps=eps)
        self.l1_loss = MaskL1Loss(eps=eps)
        self.bce_loss = BalanceLoss(
            balance_loss=balance_loss,
            main_loss_type=main_loss_type,
            negative_ratio=ohem_ratio,
        )

    def _safe(self, t):
        t = paddle.where(paddle.isnan(t), paddle.zeros_like(t), t)
        t = paddle.where(paddle.isinf(t), paddle.zeros_like(t), t)
        return t

    def forward(self, predicts, labels):
        predict_maps = predicts["maps"]
        (label_threshold_map, label_threshold_mask,
         label_shrink_map, label_shrink_mask) = labels[1:]

        label_threshold_map  = self._safe(label_threshold_map)
        label_threshold_mask = self._safe(label_threshold_mask)
        label_shrink_map     = self._safe(label_shrink_map)
        label_shrink_mask    = self._safe(label_shrink_mask)

        shrink_maps    = self._safe(predict_maps[:, 0, :, :])
        threshold_maps = self._safe(predict_maps[:, 1, :, :])
        binary_maps    = self._safe(predict_maps[:, 2, :, :])

        loss_shrink_maps    = self._safe(self.bce_loss(shrink_maps, label_shrink_map, label_shrink_mask))
        loss_threshold_maps = self._safe(self.l1_loss(threshold_maps, label_threshold_map, label_threshold_mask))
        loss_binary_maps    = self._safe(self.dice_loss(binary_maps, label_shrink_map, label_shrink_mask))

        loss_shrink_maps    = self.alpha * loss_shrink_maps
        loss_threshold_maps = self.beta  * loss_threshold_maps

        if "distance_maps" in predicts.keys():
            cbn_maps = self._safe(predicts["cbn_maps"])
            cbn_loss = self._safe(self.bce_loss(cbn_maps[:, 0, :, :], label_shrink_map, label_shrink_mask))
        else:
            cbn_loss = paddle.to_tensor([0.0])

        loss_all = loss_shrink_maps + loss_threshold_maps + loss_binary_maps
        total    = self._safe(loss_all + cbn_loss)

        return {
            "loss":                total,
            "loss_shrink_maps":    loss_shrink_maps,
            "loss_threshold_maps": loss_threshold_maps,
            "loss_binary_maps":    loss_binary_maps,
            "loss_cbn":            cbn_loss,
        }
'''

with open('/content/PaddleOCR/ppocr/losses/det_db_loss.py', 'w') as f:
    f.write(content_db)
print('✅ Patch 2/5: det_db_loss.py')

# --- PATCH 3: program.py — skip NaN/Inf batches ---
prog_file = '/content/PaddleOCR/tools/program.py'
with open(prog_file) as f:
    prog = f.read()

skip_block = (
    '                _lv = loss["loss"]\n'
    '                if paddle.isnan(_lv).any() or paddle.isinf(_lv).any():\n'
    '                    optimizer.clear_grad()\n'
    '                    continue\n'
)
target = '                loss = loss_class(preds, batch)\n'
count = prog.count(target)
if skip_block not in prog:
    prog = prog.replace(target, target + skip_block)
    with open(prog_file, 'w') as f:
        f.write(prog)
    print(f'✅ Patch 3/5: program.py (patched {count} training paths)')
else:
    print('✅ Patch 3/5: program.py (already patched)')

# --- PATCH 4: eval_det_iou.py — safe polygon validity check ---
eval_file = '/content/PaddleOCR/ppocr/metrics/eval_det_iou.py'
with open(eval_file) as f: code = f.read()
old4 = '            if not Polygon(points).is_valid:'
new4 = '''            try:
                poly = Polygon(points)
                is_valid = poly.is_valid
            except Exception:
                is_valid = False
            if not is_valid:'''
if old4 in code:
    code = code.replace(old4, new4)
    with open(eval_file, 'w') as f: f.write(code)
    print('✅ Patch 4/5: eval_det_iou.py (polygon validity)')
else:
    print('✅ Patch 4/5: eval_det_iou.py (polygon validity already patched)')

# --- PATCH 5: eval_det_iou.py — zero-division guard in IoU ---
with open(eval_file) as f: code = f.read()
old5 = '        def get_intersection_over_union(pD, pG):\n            return get_intersection(pD, pG) / get_union(pD, pG)'
new5 = '''        def get_intersection_over_union(pD, pG):
            union = get_union(pD, pG)
            if union == 0:
                return 0.0
            return get_intersection(pD, pG) / union'''
if old5 in code:
    code = code.replace(old5, new5)
    with open(eval_file, 'w') as f: f.write(code)
    print('✅ Patch 5/5: eval_det_iou.py (zero-division guard)')
else:
    print('✅ Patch 5/5: eval_det_iou.py (zero-division guard already patched)')

# --- Ensure output dir is symlinked to Drive (never overwrite symlink!) ---
import os
out_path  = '/content/workspace/output/det_finetune'
drive_out = '/content/drive/MyDrive/Rumsey_OCR/output/det_finetune'

# Remove broken symlink if Drive target disappeared
if os.path.islink(out_path) and not os.path.exists(out_path):
    os.remove(out_path)

if not os.path.exists(out_path):
    if os.path.exists(drive_out):
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        os.symlink(drive_out, out_path)
        print('✅ Output directory symlinked to Drive')
    else:
        os.makedirs(out_path, exist_ok=True)
        print('⚠️  Output is LOCAL (Drive not mounted yet — run Step 2 first!)')
else:
    if os.path.islink(out_path):
        print('✅ Output directory → Drive symlink intact')
    else:
        print('⚠️  Output is a LOCAL directory (not on Drive!)')

print('\n🚀 All 5 patches applied! Ready to run Step 8.')

## Step 8 — Train Detection Model 🔥
This is the main cell. Takes **4-6 hours**.
> The session stays alive as long as this cell is running.

> **Tip:** Keep your browser tab open (or use Colab Pro for longer sessions).


In [14]:
WORK_DIR      = '/content/workspace'
PADDLEOCR_DIR = '/content/PaddleOCR'
CONFIG        = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')

import os
os.chdir(PADDLEOCR_DIR)

# Auto-resume from latest checkpoint if it exists
latest_ckpt = os.path.join(WORK_DIR, 'output/det_finetune/latest')
if os.path.exists(latest_ckpt + '.pdparams'):
    print(f'▶️ Resuming from latest checkpoint')
    !python tools/train.py -c {CONFIG} -o Global.checkpoints={latest_ckpt}
else:
    print('🆕 Starting fresh from pretrained model')
    !python tools/train.py -c {CONFIG}

🆕 Starting fresh from pretrained model
Skipping import of the encryption module.
[2026/02/23 15:08:44] ppocr INFO: Architecture : 
[2026/02/23 15:08:44] ppocr INFO:     Backbone : 
[2026/02/23 15:08:44] ppocr INFO:         det : True
[2026/02/23 15:08:44] ppocr INFO:         name : PPLCNetV3
[2026/02/23 15:08:44] ppocr INFO:         scale : 0.75
[2026/02/23 15:08:44] ppocr INFO:     Head : 
[2026/02/23 15:08:44] ppocr INFO:         k : 50
[2026/02/23 15:08:44] ppocr INFO:         name : DBHead
[2026/02/23 15:08:44] ppocr INFO:     Neck : 
[2026/02/23 15:08:44] ppocr INFO:         name : RSEFPN
[2026/02/23 15:08:44] ppocr INFO:         out_channels : 96
[2026/02/23 15:08:44] ppocr INFO:         shortcut : True
[2026/02/23 15:08:44] ppocr INFO:     Transform : None
[2026/02/23 15:08:44] ppocr INFO:     algorithm : DB
[2026/02/23 15:08:44] ppocr INFO:     model_type : det
[2026/02/23 15:08:44] ppocr INFO: Eval : 
[2026/02/23 15:08:44] ppocr INFO:     dataset : 
[2026/02/23 15:08:44] ppocr

## Step 9 — Export Trained Model


In [ ]:
WORK_DIR      = '/content/workspace'
PADDLEOCR_DIR = '/content/PaddleOCR'
CONFIG        = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
BEST_MODEL    = os.path.join(WORK_DIR, 'output/det_finetune/best_accuracy')
INFER_OUT     = os.path.join(WORK_DIR, 'output/det_inference_finetuned')

import os
os.chdir(PADDLEOCR_DIR)

!python tools/export_model.py \
    -c {CONFIG} \
    -o Global.pretrained_model={BEST_MODEL} \
       Global.save_inference_dir={INFER_OUT}

print('Export complete!')


## Step 10 — Save Everything to Drive ✅
Saves checkpoints and the final inference model back to your Google Drive.


In [ ]:
import shutil, os
DRIVE_ROOT = '/content/drive/MyDrive/Rumsey_OCR'
WORK_DIR   = '/content/workspace'

DRIVE_OUT = os.path.join(DRIVE_ROOT, 'output')
os.makedirs(DRIVE_OUT, exist_ok=True)

# Save training checkpoints
src_ckpt = os.path.join(WORK_DIR, 'output/det_finetune')
dst_ckpt = os.path.join(DRIVE_OUT, 'det_finetune')
if os.path.exists(src_ckpt):
    shutil.copytree(src_ckpt, dst_ckpt, dirs_exist_ok=True)
    print(f'✅ Checkpoints saved to Drive')

# Save inference model
src_inf = os.path.join(WORK_DIR, 'output/det_inference_finetuned')
dst_inf = os.path.join(DRIVE_OUT, 'det_inference_finetuned')
if os.path.exists(src_inf):
    shutil.copytree(src_inf, dst_inf, dirs_exist_ok=True)
    print(f'✅ Inference model saved to Drive')

print(f'\nAll outputs at: {DRIVE_OUT}')
print('Download det_inference_finetuned/ to your laptop when ready.')


## ✅ All Done!

After the notebook completes:
1. Download `output/det_inference_finetuned/` from Drive to your laptop
2. Place it in `Rumsey_Map_OCR/output/det_inference_finetuned/`
3. Run `step4_evaluate_and_infer.py` on your laptop to see the improved results

**Expected improvement: 27% → 50-60%+ recall** 🎯
